In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.metrics import root_mean_squared_error as root_mse
import os
from tqdm import tqdm
import plotly.express as px
import seaborn as sns
from joblib import Parallel, delayed
from scipy.spatial import ConvexHull
from tools import (element_content, T_delta_Sconf, compositionTranform, simplexBasis, formula2string, unique_elements,
                          getEhull, readVASPresults, getElementCount, styler, recoverFormulaFromVac)

In [ ]:
inference_file = 'data/inference2_ccs-gnn_predictions_subset.pkl.gz'
full_inference = pd.read_pickle(inference_file)
models_mapper = {'allegro_aflow_ccs_new2': 'allegro_aflow_ccs-Pmm2+', 'allegro_random_ccs_new2': 'allegro_random_ccs-Pmm2+',
                 'nequip_aflow_ccs_new2': 'nequip_aflow_ccs-Pmm2+', 'nequip_random_ccs_new2': 'nequip_random_ccs-Pmm2+',
                 'allegro_aflow_ccs_all': 'allegro_aflow_ccs', 'allegro_random_ccs_all': 'allegro_random_ccs',
                 'nequip_aflow_ccs_all': 'nequip_aflow_ccs', 'nequip_random_ccs_all': 'nequip_random_ccs',
                 'allegro_aflow_ccs_P1limit': 'allegro_aflow_ccs-C2+', 'allegro_random_ccs_P1limit': 'allegro_random_ccs-C2+', 
                 'nequip_aflow_ccs_P1limit': 'nequip_aflow_ccs-C2+', 'nequip_random_ccs_P1limit': 'nequip_random_ccs-C2+'}
full_inference.rename(mapper=models_mapper, axis=1, inplace=True)
fi_available_models = [m for m in full_inference.columns if (m.startswith('allegro') or m.startswith('nequip'))]
for model in fi_available_models:
    full_inference[model] /= full_inference['natoms']    
for element in ['W', 'Se', 'U', 'Np']:
    full_inference[f'{element}_nsites'] = full_inference['formula'].apply(lambda x: x[element])
full_inference[f'defect_nsites'] = 48 - full_inference['W_nsites'] - full_inference['Se_nsites']

print(full_inference.shape)
full_inference.groupby('type').size()

For entries of the second inference CCS, plot pairwise relationships between the compositions, space groups, weights, and number of defect positions in a 4×4×1 supercell. W_nsites and Se_nsites denote the number of positions occupied by W and Se, respectively. The fictitious U_nsites and Np_nsites atomic types correspond to the number of vacancies on the Me and X sublattice (0 or 1 by the design of the second inference CCS), respectively.

In [ ]:
sns.pairplot(full_inference[['W_nsites', 'U_nsites', 'Se_nsites', 'Np_nsites', 'space_group_no', 'type', 
                             'weight', 'defect_nsites']], hue='type', corner=True)
plt.tight_layout()
plt.show()

Evaluate $E_{hull}$ for each composition within the 3D compositional simplex.

In [ ]:
neat_elements = pd.read_pickle("data/neat_elements-dft_subset.pkl.gz")
result = pd.read_pickle('data/train_ccs-gnn_predictions_dft_subset.pkl.gz')
simplex_basis = simplexBasis(len(unique_elements) - 1)
simplex_basis_df = pd.DataFrame(simplex_basis, columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_basis_df['formation_energy_pa'] = 0
simplex_basis_df['formula'] = neat_elements.loc[list(unique_elements)]['formula'].values
simplex_basis_df['formula_str'] = simplex_basis_df['formula'].apply(formula2string)
simplex_basis_df['compound'] = unique_elements
simplex_basis_df['part'] = 'Neat elements'
simplex_basis_df['size'] = 6

# DFT-derived structures
print('Collecting DFT-derived structures...')
compositions = []
for i in tqdm(result.index):
    compositions += [[element_content(element, result.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_dft = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                  columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_dft['formation_energy_pa'] = result['formation_energy_per_atom'].values
simplex_coords_dft['formula'] = result['formula'].values
simplex_coords_dft['formula_str'] = simplex_coords_dft['formula'].apply(formula2string)
simplex_coords_dft['compound'] = result['compound'].values
simplex_coords_dft['part'] = result['Space_group_no'].apply(lambda x: 'CCS (Pmm2 and higher)' if x > 8 else 'CCS (Cm and lower)').values
simplex_coords_dft['size'] = result['Space_group_no'].apply(lambda x: (4 if x > 8 else 1)).values

# 2DMD dataset
path_2dmd = 'data/2d-materials-point-defects-all-table.pkl.gz'
data_2dmd = pd.read_pickle(path_2dmd)
data_2dmd = data_2dmd[(data_2dmd['base'] == 'WSe2') | (data_2dmd['base'] == 'MoS2')].copy()
data_2dmd['natoms'] = data_2dmd['atomic_numbers'].apply(len)
data_2dmd['formation_energy_per_atom'] = data_2dmd['formation_energy'] / data_2dmd['natoms']
compositions = []
print('Collecting structures of the 2DMD dataset...')
for i in tqdm(data_2dmd.index):
    compositions += [[element_content(element, data_2dmd.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_2dmd = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                   columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_2dmd['formation_energy_pa'] = data_2dmd['structure_formation_energy_per_atom'].values
simplex_coords_2dmd['formula'] = data_2dmd['formula'].values
simplex_coords_2dmd['formula_str'] = simplex_coords_2dmd['formula'].apply(formula2string)
simplex_coords_2dmd['compound'] = data_2dmd.index
simplex_coords_2dmd['part'] = '2DMD datasets'
simplex_coords_2dmd['size'] = .3

simplex_df = pd.concat((simplex_basis_df, simplex_coords_dft, simplex_coords_2dmd), axis=0)
simplex_df.set_index('compound', inplace=True, drop=True)
available_energies_indexes = simplex_df.index
available_parts = simplex_df['part'].unique()
points = simplex_df.iloc[:, :len(unique_elements)].values
hull = ConvexHull(points)

In [ ]:
simplex_file = 'data/inference2_ccs-gnn_predictions_subset-simplex_coord.pkl.gz'
if os.path.exists(simplex_file):
    print('Reading simplex from simplex_file')
    simplex_coords_fi1 = pd.read_pickle(simplex_file)
else:
    compositions = []
    print('Collecting structures of the full_inference dataset...')
    for i in tqdm(full_inference.index):
        compositions += [[element_content(element, full_inference.loc[i], relative=False) for element in unique_elements]]
    compositions = np.array(compositions)
    simplex_basis = simplexBasis(len(unique_elements) - 1)
    simplex_coords_fi1 = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                      columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)], 
                                      index=full_inference.index)    
    for model in fi_available_models:
        simplex_coords_fi1[f'formation_energy_pa_{model}'] = full_inference[model]
    simplex_coords_fi1['formula'] = full_inference['formula']
    simplex_coords_fi1['formula_str'] = full_inference['formula_str']
    simplex_coords_fi1['compound'] = full_inference.index
    simplex_coords_fi1['part'] = 'Inference 1 (full CCS)'
    simplex_coords_fi1['size'] = 2
    
    points_f1 = simplex_coords_fi1.iloc[:, :len(unique_elements)].values
    simplex_coords_fi1['Reference_hull_E'] = Parallel(n_jobs=-1)(delayed(getEhull)(point, hull) for point in tqdm(points_f1[:, :3]))

    for model in fi_available_models:
        simplex_coords_fi1[f'E_above_hull_{model}'] = simplex_coords_fi1[f'formation_energy_pa_{model}'] - simplex_coords_fi1['Reference_hull_E']
        print(model, simplex_coords_fi1[f'E_above_hull_{model}'].min())
    energies_df = pd.read_pickle('data/train_ccs-2dmd-convex_hull.pkl.gz')
    simplex_coords_fi1 = pd.concat((energies_df[energies_df['part'] == 'Neat elements'], simplex_coords_fi1), axis=0)
    simplex_coords_fi1.drop(['E_above_hull', 'DefineHull', '<25meV/atom_above', 'compound'], axis=1, inplace=True)    
    simplex_coords_fi1.to_pickle(simplex_file)

print(simplex_coords_fi1.columns)
print(simplex_coords_fi1.shape)

In [ ]:
list_of_Ehull_models = [m for m in simplex_coords_fi1.columns if (m.startswith('E_above') & (re.match('.*_random_.*', m) is not None))]
print('list_of_Ehull_models:', len(list_of_Ehull_models))
list_of_available_formulas = simplex_coords_fi1[simplex_coords_fi1['part'] != 'Neat elements']
list_of_available_formulas = list_of_available_formulas['formula_str'].unique()
print('list_of_available_formulas:', len(list_of_available_formulas))

In [ ]:
# Select and write structures for VASP (version for all available formulas)

complexity = 3  # maximum number of the most favorable configurations per model
ht2_structures = []

for formula_ in tqdm(list_of_available_formulas):
    subset = simplex_coords_fi1[simplex_coords_fi1['formula_str'] == formula_]
    for model_ in list_of_Ehull_models:
        ht2_structures += list(subset.sort_values(model_).index[:complexity])

print(len(ht2_structures), 'structures selected with', len(set(ht2_structures)), 'unique')
ht2_structures = list(set(ht2_structures))
simplex_coords_fi1_subset = simplex_coords_fi1.loc[ht2_structures]

For 11,486 entries of HT#2 (i.e., most energetically favorable structure realizations as predicted using the GNN models developed), plot comparison of GNN-predicted $E_F$ values.

In [ ]:
modes_all = ['2dmd-ldc', '2dmd-hdc', '2dmd', 'ccs-Pmm2+', 'ccs-C2+', 'ccs']
mode_color = dict(zip(modes_all, ['b', 'g', 'r', 'c', 'm', 'grey']))
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), sharex=True, sharey=True)
for archit_id, archit in enumerate(['allegro', 'nequip']):
    for pretrain_id, pretrain in enumerate(['aflow', 'random']):
        axs[1, pretrain_id].set_xlabel('$E_{F}$, eV/atom')
        axs[archit_id, 0].set_ylabel('Number of configurations')
        FE_available_models_of_archit = [f'formation_energy_pa_{archit}_{pretrain}_{mode}' for mode in modes_all]
        axs[archit_id, pretrain_id].set_title(f'{archit}_{pretrain}')
        axs[archit_id, pretrain_id].hist([simplex_coords_fi1_subset[model] for model in FE_available_models_of_archit], 
                                         stacked=True, edgecolor='grey', label=modes_all, color=list(map(lambda x: mode_color[x], modes_all)), 
                                         bins=np.linspace(-0.9, -0.4, 51))
        axs[archit_id, pretrain_id].grid(alpha=0.3)          
axs[archit_id, pretrain_id].legend(title='Training data')
plt.tight_layout()
plt.show()

For 11,486 entries of HT#2 (i.e., most energetically favorable structure realizations as predicted using the GNN models developed), plot comparison of $E_{hull}$ values obtained from GNN-predicted $E_F$ values.

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), sharex=True, sharey=True)
for archit_id, archit in enumerate(['allegro', 'nequip']):
    for pretrain_id, pretrain in enumerate(['aflow', 'random']):
        axs[1, pretrain_id].set_xlabel('$E_{hull}$, eV/atom')
        axs[archit_id, 0].set_ylabel('Number of configurations')
        Ehull_available_models_of_archit = [f'E_above_hull_{archit}_{pretrain}_{mode}' for mode in modes_all]
        axs[archit_id, pretrain_id].set_title(f'{archit}_{pretrain}')
        axs[archit_id, pretrain_id].hist([simplex_coords_fi1_subset[model] for model in Ehull_available_models_of_archit],
                                         stacked=True, edgecolor='grey', label=modes_all, color=list(map(lambda x: mode_color[x], modes_all)), 
                                         bins=np.linspace(-50e-3, 350e-3, 41))
        axs[archit_id, pretrain_id].grid(alpha=0.3)          
axs[archit_id, pretrain_id].legend(title='Training data')
plt.tight_layout()
plt.show()

In [ ]:
data_group = 'WMoUSeSNp'
structures_fi_result = readVASPresults(data_group, data_subgroup='enh2_low_en', return_neat_elements=False)
structures_fi_result.drop('formula', axis=1, inplace=True)  # avoid multiple formulas, ccs-based is preferred
structures_fi_result = pd.concat((structures_fi_result.loc[simplex_coords_fi1_subset.index],  # only selected structures
                                  simplex_coords_fi1_subset), axis=1)
structures_fi_result['E_above_hull'] = structures_fi_result['formation_energy_per_atom'] - structures_fi_result['Reference_hull_E']

for element in ['S', 'Se', 'Mo', 'W']:
    structures_fi_result[element] = [getElementCount(structures_fi_result.loc[i], element) for i in structures_fi_result.index]
structures_fi_result['meVac'] = 16 - structures_fi_result['Mo'] - structures_fi_result['W']
structures_fi_result['xVac'] = 32 - structures_fi_result['S'] - structures_fi_result['Se']

simplex_coords_fi1_subset_finished = simplex_coords_fi1_subset.loc[structures_fi_result.index].copy()
simplex_coords_fi1_subset_finished['E_above_hull'] = structures_fi_result['E_above_hull'].values

Plot heatmap of $E_F$ assessment errors obtained on the HT#2 structures depending on training/fine-tuning data and pretraining states of the used GNNs.

In [ ]:
list_of_Eform_models = [m for m in simplex_coords_fi1.columns if (m.startswith('formation_energy_pa_'))]
scores = pd.DataFrame([(model, 1000 * root_mse(structures_fi_result['formation_energy_per_atom'], structures_fi_result[model])) 
                       for model in list_of_Eform_models], columns=['model', 'rmse,meV/atom'])
scores['architecture'] = scores['model'].apply(lambda x: 'Allegro' if x.startswith('formation_energy_pa_allegro') else 'NequIP')
scores['pre-training'] = scores['model'].apply(lambda x: re.match('.*random.*', x) is None)
scores['pre-training data'] = scores['pre-training'].apply(lambda x: '\n(AflowLib)' if x else '')
scores['data used'] = scores['model'].apply(lambda x: x.split('random_')[1] if re.match('.*random.*', x) is not None else x.split('aflow_')[1])
scores['model'] = scores['architecture'] + scores['pre-training data']
scores = scores[sorted(scores.columns)]
scores = scores[scores['data used'] != 'ccs_more_than_half_Me_tt']
scores['data used'] = scores['data used'].apply(lambda x: x if x != 'ccs_more_than_half_Me' else 'ccs')

plt.figure(figsize=(4.8, 3.98))
ax = sns.heatmap(scores.pivot(index='model', columns='data used', values='rmse,meV/atom'),
                 vmin=5, vmax=65, annot=True, fmt=".1f", cbar_kws={'label': 'RMSE, meV/atom', 'drawedges': False, 'shrink': 0.9})
ax.set(xlabel='Training set', ylabel='Architecture (pretraining)')
ax.set_title(f'HT#2 ({structures_fi_result.shape[0]} structures)')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Plot heatmap of $E_{hull}$ assessment errors obtained on the HT#2 structures depending on training/fine-tuning data and pretraining states of the used GNNs.

In [ ]:
list_of_Ehull_models = [m for m in simplex_coords_fi1.columns if (m.startswith('E_above_hull_'))]
scores = pd.DataFrame([(model, 1000 * root_mse(structures_fi_result['E_above_hull'], structures_fi_result[model])) 
                       for model in list_of_Ehull_models], columns=['model', 'rmse,meV/atom'])
scores['architecture'] = scores['model'].apply(lambda x: 'Allegro' if x.startswith('E_above_hull_allegro') else 'NequIP')
scores['pre-training'] = scores['model'].apply(lambda x: re.match('.*random.*', x) is None)
scores['pre-training data'] = scores['pre-training'].apply(lambda x: '\n(AflowLib)' if x else '')
scores['data used'] = scores['model'].apply(lambda x: x.split('random_')[1] if re.match('.*random.*', x) is not None else x.split('aflow_')[1])
scores['model'] = scores['architecture'] + scores['pre-training data']
scores = scores[sorted(scores.columns)]
scores = scores[scores['data used'] != 'ccs_more_than_half_Me_tt']
scores['data used'] = scores['data used'].apply(lambda x: x if x != 'ccs_more_than_half_Me' else 'ccs')

ax = sns.heatmap(scores.pivot(index='model', columns='data used', values='rmse,meV/atom'),
                 vmin=5, vmax=65, annot=True, fmt=".2f", cbar_kws={'label': 'RMSE, meV/atom', 'drawedges': False, 'shrink': 0.9})
ax.set(xlabel='Training set', ylabel='Architecture (pre-training)')
formula = re.sub('[\d]', lambda x: f'$_{x.group()}$', 'WSe2')
ax.set_title(f'holdout test #3')
plt.show()

Plot comparison of the GNN-predicted and DFT-derived $E_F$ values.

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=6, figsize=(15, 11), sharex=True, sharey=True)
for arch_index, arch_ in enumerate(['allegro', 'nequip']):    
    for pr_index_, pretrain_set_ in enumerate(['random', 'aflow']):         
        axs[2 * arch_index + pr_index_, 0].set_ylabel('DFT-derived $E_{F}$, eV/atom')        
        for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'ccs-Pmm2+']):
            model_ = f'{arch_}_{pretrain_set_}_{train_set_}'
            axs[2 * arch_index + pr_index_, tr_index_].set_title(model_)
            axs[2 * arch_index + pr_index_, tr_index_].scatter(structures_fi_result[f'formation_energy_pa_{model_}'], 
                                                               structures_fi_result['formation_energy_per_atom'], alpha=0.2)
            axs[2 * arch_index + pr_index_, tr_index_].plot([-1.0, -.4], [-1.0, -.4], 'k', alpha=0.3)
            axs[2 * arch_index + pr_index_, tr_index_].grid(alpha=0.3)
for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'cc+Pmm2+']):
    axs[-1, tr_index_].set_xlabel('GNN-predicted $E_{F}$, eV/atom')
plt.tight_layout()
plt.show()

Plot comparison of the recalculated GNNs’ predictions and DFT-derived $E_{hull}$ values.

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=6, figsize=(15, 11), sharex=True, sharey=True)
for arch_index, arch_ in enumerate(['allegro', 'nequip']):    
    for pr_index_, pretrain_set_ in enumerate(['random', 'aflow']):        
        axs[2 * arch_index + pr_index_, 0].set_ylabel('DFT-derived $E_{hull}$, eV/atom')        
        for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'ccs-Pmm2+']):
            model_ = f'{arch_}_{pretrain_set_}_{train_set_}'
            axs[2 * arch_index + pr_index_, tr_index_].set_title(model_)
            axs[2 * arch_index + pr_index_, tr_index_].scatter(structures_fi_result[f'E_above_hull_{model_}'], 
                                                               structures_fi_result['E_above_hull'], alpha=0.2)
            axs[2 * arch_index + pr_index_, tr_index_].plot([-.05, .35], [-.05, .35], 'k', alpha=0.3)
            axs[2 * arch_index + pr_index_, tr_index_].grid(alpha=0.3)
for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'cc+Pmm2+']):
    axs[-1, tr_index_].set_xlabel('$E_{hull}$ from GNN $E_{F}$, eV/atom')
plt.tight_layout()
plt.show()

Plot $E_{hull}$ (color) for each composition within the 3D compositional simplex.

In [ ]:
size_mult = 11
df_energies_on_hull = simplex_coords_fi1_subset.copy()
df_energies_on_hull['E_above_hull'] = structures_fi_result['E_above_hull']
df_energies_on_hull['Reference_hull_E'] = structures_fi_result['Reference_hull_E']
df_energies_on_hull = pd.concat((simplex_coords_fi1.iloc[:4], df_energies_on_hull), axis=0)
df_energies_on_hull['size'] = -df_energies_on_hull['E_above_hull']
df_energies_on_hull['size'] -= df_energies_on_hull['size'].min()
df_energies_on_hull['size'] = size_mult * (df_energies_on_hull['size'] + 0.25) 
df_energies_on_hull['size'] = df_energies_on_hull['size'].fillna(25)

fig = px.scatter_3d(df_energies_on_hull, x='SC0', y='SC1', z='SC2', color=f'E_above_hull',
                    opacity=1, size='size', hover_data=[df_energies_on_hull['part'], df_energies_on_hull['E_above_hull'], df_energies_on_hull.index], 
                    color_continuous_scale='inferno')
fig.update_traces(marker=dict(line=dict(width=0,)), selector=dict(mode='markers'))
fig.show()

In [ ]:
if size_mult == 11:
    camera = dict(up=dict(x=0, y=0, z=1), eye=dict(x=1.5, y=0.5, z=0.5))
    fig.update_layout(scene_camera=camera)
    fig.show()
else:
    camera = dict(up=dict(x=0, y=0, z=1), eye=dict(x=.44, y=0.07, z=0.19))
    fig.update_layout(scene_camera=camera)
    fig.show()

Plot distribution of DFT-derived $E_{hull}$ by vacancy type (2 meV/atom bins).

In [ ]:
Me_X_vac_2color = {'0 : 0': 'k', '1 : 0': 'orange', '0 : 1': 'purple'}
vmevx = 'V$_{Me}$:V$_{X}$'
labels = [' : '.join(map(str, Me_X_vac)) for Me_X_vac in [(0, 0), (0, 1), (1, 0)]]
colors = [Me_X_vac_2color[' : '.join(map(str, Me_X_vac))] for Me_X_vac in [(0, 0), (0, 1), (1, 0)]]
plt.figure(figsize=(5, 3))
plt.hist([1000 * structures_fi_result[(structures_fi_result['meVac'] == Me_X_vac[0]) & (structures_fi_result['xVac'] == Me_X_vac[1])]['E_above_hull'] 
          for Me_X_vac in [(0, 0), (0, 1), (1, 0)]], label=labels, color=colors, stacked=True, bins=np.linspace(-2, 124, 64), edgecolor='grey')
plt.legend(title=vmevx, loc=1, labelspacing=0.3)
plt.ylabel('Number of HT#2 entries')
plt.xlabel('DFT-derived $E_{hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
print(sum(structures_fi_result['E_above_hull'] < 0), 'structures below the hull')
print('with maximum drop of ', 1000 * structures_fi_result['E_above_hull'].min(), 'meV/atom')
print(structures_fi_result[structures_fi_result['E_above_hull'] < 0].groupby('nelements').size())
print(structures_fi_result[structures_fi_result['E_above_hull'] < 0].groupby('nelements')['E_above_hull'].min())

### Configurational entropy impact

In [ ]:
temperatures = np.linspace(0, 2000, 21)
print('List of temperatures used: ', temperatures)

In [ ]:
deltaG_table = []
formulas = full_inference['formula_str'].unique()

for form in formulas:
    subset = full_inference[full_inference['formula_str'] == form]
    ref_hull = df_energies_on_hull[df_energies_on_hull['formula_str'] == form]['Reference_hull_E'][0]
    deltaG_table += [[T_delta_Sconf(subset['nequip_random_ccs'] * subset['natoms'], subset['weight'], t_, subset['natoms'][0]) - ref_hull
                      for t_ in temperatures]]    
deltaG_table = pd.DataFrame(deltaG_table, index=formulas, columns=list(temperatures))
deltaG_table['∆Gdrop,eV/atom'] = deltaG_table[temperatures[-1]] - deltaG_table[temperatures[0]]
deltaG_table['type'] = full_inference.groupby('formula_str').first()['type']
deltaG_table['numStrBefMerge'] = full_inference.groupby('formula_str')['weight'].sum()
deltaG_table['numStrAftMerge'] = full_inference.groupby('formula_str').size()
deltaG_table['formula'] = full_inference.groupby('formula_str').first()['formula']
deltaG_table['meVac'] = deltaG_table['formula'].apply(lambda x: x['U'])
deltaG_table['xVac'] = deltaG_table['formula'].apply(lambda x: x['Np'])
deltaG_table['S'] = deltaG_table['formula'].apply(lambda x: x['S'])
deltaG_table['Se'] = deltaG_table['formula'].apply(lambda x: x['Se'])
deltaG_table['Mo'] = deltaG_table['formula'].apply(lambda x: x['Mo'])
deltaG_table['W'] = deltaG_table['formula'].apply(lambda x: x['W'])
deltaG_table['DFTminEhull'] = structures_fi_result.groupby('formula_str')['E_above_hull'].min()
deltaG_table.head()

In [ ]:
Me_X_vac_2color = {'0 : 0': 'k', '1 : 0': 'orange', '0 : 1': 'purple'}
vmevx = 'V$_{Me}$:V$_{X}$'
super_titles = ['all', 'complete structure sets', 'highest space group only']

# Vacancy-based visual for less60k and !less60k
for id_, res_table_plot in enumerate([deltaG_table, 
                                      deltaG_table[deltaG_table['type'] == 'less60k'],
                                      deltaG_table[deltaG_table['type'] != 'less60k']]):
    N_formulas = res_table_plot.shape[0]
    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(8, 5), sharey='row', gridspec_kw={'height_ratios': [3, 1]})
    plt.suptitle(f'{super_titles[id_]}: {N_formulas} compositions', y=0.95)
    for Me_X_vac_id, Me_X_vac in enumerate([(0, 0), (0, 1), (1, 0)]):
        tag = ' : '.join(map(str, Me_X_vac))
        subset = res_table_plot[(res_table_plot['meVac'] == Me_X_vac[0]) & (res_table_plot['xVac'] == Me_X_vac[1])]
        for form in subset.index:
            axs[0, Me_X_vac_id].plot(temperatures, 1000 * res_table_plot.loc[form][temperatures], color=Me_X_vac_2color[tag], alpha=0.2, lw=1)
        axs[0, Me_X_vac_id].scatter(np.zeros(subset.shape[0]), 1000 * subset['DFTminEhull'], color=Me_X_vac_2color[tag], marker='x')
        axs[0, Me_X_vac_id].set_title(f'{vmevx} = {tag}')
        axs[0, Me_X_vac_id].grid(alpha=0.3)
        axs[0, Me_X_vac_id].set_xlabel('T, K')
        axs[1, Me_X_vac_id].hist(1000 * subset['∆Gdrop,eV/atom'], bins=1000 * np.linspace(-0.06, 0, 25), color=Me_X_vac_2color[tag], edgecolor='grey')
        axs[1, Me_X_vac_id].grid(alpha=0.3)
        axs[1, Me_X_vac_id].set_xlabel('${\Delta}G$ drop (0-2000K), meV/atom')
    axs[0, 0].set_ylabel('${\Delta}G$ with $T{\Delta}S_{conf}$\nabove 0 K hull, meV/atom')
    axs[1, 0].set_ylabel('Number of\ncompositions')
    axs[0, 0].set_ylim((-70, 100))
    plt.tight_layout()
    plt.show()

In [ ]:
sns.pairplot(structures_fi_result[(structures_fi_result['meVac'] == 1) &
                                  ((structures_fi_result['S'] > 30) | (structures_fi_result['Se'] > 30)) &
                                  ((structures_fi_result['Mo'] > 14) | (structures_fi_result['W'] > 14))][['E_above_hull', 'S', 'Mo']])

### 0 K $E_{hull}$

#### Me vacancy, Mo-W alloys

Plot DFT-derived $E_{hull}$ for the most favorable GNN-predicted configurations with $V_{Me}$ or $V_X$ across binary and ternary systems and minimal shifts toward quaternary compositions. For Se/S alloys, dashed lines mark structures that were undersampled, retaining only the highest-symmetry space groups. GNN screening used models trained from scratch in this study.

In [ ]:
as_frame = structures_fi_result[(structures_fi_result['meVac'] == 1)].groupby(['S', 'Se', 'Mo', 'W']).\
                                                                      min('E_above_hull')[['E_above_hull_nequip_random_ccs', 'E_above_hull']]
plt.figure(figsize=(4, 4))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['c', 'm', 'y', 'k'])
for index_0 in [31, 32, 0, 1]:
    for index_1 in set(map(lambda x: x[0], as_frame.loc[index_0].index)):
        Mo_cont = list(map(lambda x: x[0], as_frame.loc[index_0].loc[index_1].index))
        if len(Mo_cont) > 4:
            if index_0 == 0:
                label = 'Mo$_{x}$W$_{15-x}$Se$_{' + str(index_1) + '}$'
            elif index_1 == 0:
                label = 'Mo$_{x}$W$_{15-x}$S$_{' + str(index_0) + '}$'
            else:
                label = 'Mo$_{x}$W$_{15-x}$S$_{' + str(index_0) + '}$Se$_{' + str(index_1) + '}$'
            plt.plot(Mo_cont, 1000 * as_frame.loc[index_0, index_1]['E_above_hull'], '.-', lw=1.5, label=label)     
plt.legend(loc='upper left', labelspacing=0.1)
plt.ylim((72.5, 102.5))
plt.xlabel('x')
plt.xlim((-1, 17))
plt.xticks(ticks=np.linspace(0, 16, 9, dtype=int))
plt.ylabel('$E_{hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.text(-5, 102, '(a)', size=16)
plt.tight_layout()
plt.show()
plt.rcParams['axes.prop_cycle'] = plt.rcParamsDefault['axes.prop_cycle']

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(as_frame['E_above_hull_nequip_random_ccs'], as_frame['E_above_hull'])
plt.plot([0.055, 0.100], [0.055, 0.100], alpha=.2)
plt.grid(alpha=0.3)
plt.show()

#### X vacancy, Mo-W alloys

In [ ]:
as_frame = structures_fi_result[(structures_fi_result['xVac'] == 1)].groupby(['S', 'Se', 'Mo', 'W']).\
                                                                     min('E_above_hull')[['E_above_hull_nequip_random_ccs', 'E_above_hull']]
plt.figure(figsize=(4, 4))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['c', 'm', 'y', 'k'])
for index_0 in [30, 31, 0, 1]:
    for index_1 in set(map(lambda x: x[0], as_frame.loc[index_0].index)):
        Mo_cont = list(map(lambda x: x[0], as_frame.loc[index_0].loc[index_1].index))
        if len(Mo_cont) > 4:
            if index_0 == 0:
                label = 'Mo$_{x}$W$_{16-x}$Se$_{' + str(index_1) + '}$'
            elif index_1 == 0:
                label = 'Mo$_{x}$W$_{16-x}$S$_{' + str(index_0) + '}$'
            else:
                label = 'Mo$_{x}$W$_{16-x}$S$_{' + str(index_0) + '}$Se$_{' + str(index_1) + '}$'
            plt.plot(Mo_cont, 1000 * as_frame.loc[index_0, index_1]['E_above_hull'], '.-', lw=1.5, label=label)     
plt.legend(loc='lower left', labelspacing=0.1)
plt.ylim((26, 39))
plt.xlabel('x')
plt.xlim((-1, 17))
plt.xticks(ticks=np.linspace(0, 16, 9, dtype=int))
plt.ylabel('$E_{hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.text(-5, 39, '(c)', size=16)
plt.tight_layout()
plt.show()
plt.rcParams['axes.prop_cycle'] = plt.rcParamsDefault['axes.prop_cycle']

#### Me vacancy, S-Se alloy

In [ ]:
as_frame = structures_fi_result[(structures_fi_result['meVac'] == 1)].groupby(['Mo', 'W', 'S', 'Se']).\
                                                                      min('E_above_hull')[['E_above_hull_nequip_random_ccs', 'E_above_hull']]
inference_type_mapping = full_inference[['formula_str', 'type']].groupby('formula_str').first()['type']
forms_of_as_frame = list(map(lambda x: f'W{x[1]}Mo{x[0]}Se{x[3]}S{x[2]}', as_frame.index))
as_frame['type'] = inference_type_mapping.loc[forms_of_as_frame].values
as_frame['style'] = styler(as_frame)

plt.figure(figsize=(4, 4))
for style_it in range(3):
    plt.gca().set_prop_cycle(None)
    for index_0 in set(map(lambda x: x[0], as_frame.index)):
        for index_1 in set(map(lambda x: x[0], as_frame.loc[index_0].index)):
            S_cont = list(map(lambda x: x[0], as_frame.loc[index_0].loc[index_1].index))
            if len(S_cont) > 4:
                if index_0 == 0:
                    label = 'W$_{' + str(index_1) + '}$S$_x$Se$_{32-x}$'
                elif index_1 == 0:
                    label = 'Mo$_{' + str(index_0) + '}$S$_x$Se$_{32-x}$'
                else:
                    label = 'Mo$_{' + str(index_0) + '}$W$_{' + str(index_1) + '}$S$_x$Se$_{32-x}$'
                as_frame_sub = as_frame.loc[index_0, index_1]
                unique_styles = sorted(as_frame_sub['style'].unique())
                mask = as_frame_sub['style'] == unique_styles[style_it] if style_it != 1 else np.full_like(as_frame_sub['style'], True, dtype=bool)
                plt.plot(np.array(S_cont)[mask], 1000 * as_frame_sub[mask]['E_above_hull'], '.-' if style_it != 1 else '--',
                         lw='1.5' if style_it != 1 else 1, label=label if style_it == 0 else None)
plt.ylim((72.5, 102.5))
plt.legend(loc='upper left', labelspacing=0.1)
plt.xlabel('x')
plt.xlim((-1, 33))
plt.xticks(ticks=np.linspace(0, 32, 9, dtype=int))
plt.ylabel('$E_{hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.text(-9, 102, '(b)', size=16)
plt.tight_layout()
plt.show()

#### X vacancy, S-Se alloy

In [ ]:
as_frame = structures_fi_result[(structures_fi_result['xVac'] == 1)].groupby(['Mo', 'W', 'S', 'Se']).\
                                                                     min('E_above_hull')[['E_above_hull_nequip_random_ccs', 'E_above_hull']]
inference_type_mapping = full_inference[['formula_str', 'type']].groupby('formula_str').first()['type']
forms_of_as_frame = list(map(lambda x: f'W{x[1]}Mo{x[0]}Se{x[3]}S{x[2]}', as_frame.index))
as_frame['type'] = inference_type_mapping.loc[forms_of_as_frame].values
as_frame['style'] = styler(as_frame)

plt.figure(figsize=(4, 4))
for style_it in range(3):
    plt.gca().set_prop_cycle(None)
    for index_0 in set(map(lambda x: x[0], as_frame.index)):
        for index_1 in set(map(lambda x: x[0], as_frame.loc[index_0].index)):
            S_cont = list(map(lambda x: x[0], as_frame.loc[index_0].loc[index_1].index))
            if len(S_cont) > 4:
                if index_0 == 0:
                    label = 'W$_{' + str(index_1) + '}$S$_x$Se$_{31-x}$'
                elif index_1 == 0:
                    label = 'Mo$_{' + str(index_0) + '}$S$_x$Se$_{31-x}$'
                else:
                    label = 'Mo$_{' + str(index_0) + '}$W$_{' + str(index_1) + '}$S$_x$Se$_{31-x}$'
                as_frame_sub = as_frame.loc[index_0, index_1]
                unique_styles = sorted(as_frame_sub['style'].unique())
                mask = as_frame_sub['style'] == unique_styles[style_it] if style_it != 1 else np.full_like(as_frame_sub['style'], True, dtype=bool)
                plt.plot(np.array(S_cont)[mask], 1000 * as_frame_sub[mask]['E_above_hull'], '.-' if style_it != 1 else '--',
                         lw='1.5' if style_it != 1 else 1, label=label if style_it == 0 else None)    
plt.ylim((25, 39))
plt.legend(loc='lower left', labelspacing=0.1)
plt.xlabel('x')
plt.xlim((-1, 33))
plt.xticks(ticks=np.linspace(0, 32, 9, dtype=int))
plt.ylabel('$E_{hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.text(-8, 39, '(d)', size=16)
plt.tight_layout()
plt.show()

### Particular compositions

Plot temperature dependencies of free energy of formation above the finite-temperature convex hulls, $∆G_{hull}$, across the compositions of the second inference CCS. 

In [ ]:
res_table_plot = deltaG_table[deltaG_table['type'] == 'less60k']  # fix complete compositions only
alloys = ['MoWS2', 'MoWSe2', 'WSSe', 'MoSSe']
Me_X_vac_2color = {'w/o vacancies': 'k', 'with S/Se vacancy': 'purple'}
alloy_formulas = ['Mo$\mathbf{_x}$W$\mathbf{_{1-x}}$S$\mathbf{_2}$', 'Mo$\mathbf{_x}$W$\mathbf{_{1-x}}$Se$\mathbf{_2}$',
                  'WS$\mathbf{_x}$Se$\mathbf{_{2-x}}$', 'MoS$\mathbf{_x}$Se$\mathbf{_{2-x}}$']
alloy_subset_masks = [res_table_plot['S'] == 31, res_table_plot['Se'] == 31,
                      res_table_plot['Mo'] == 16, res_table_plot['W'] == 16]
elements_to_recover_sets = [['S'], ['Se'], ['S', 'Se'], ['S', 'Se']]
for id_, alloy in enumerate(alloys):    
    subset = res_table_plot[alloy_subset_masks[id_]]
    elements_to_recover = elements_to_recover_sets[id_]
    xVac_formulas = subset.index
    xVac_formulas_recovered = []
    inverse_mask = []
    for form in xVac_formulas:
        form_dict = res_table_plot.loc[form]['formula']
        xVac_formulas_recovered_add = recoverFormulaFromVac(form_dict, elements_to_recover)
        xVac_formulas_recovered_add = [formula for formula in xVac_formulas_recovered_add 
                                       if formula in deltaG_table[deltaG_table['type'] == 'less60k'].index]
        if len(xVac_formulas_recovered_add) > 0:
            inverse_mask += [True]
            xVac_formulas_recovered += xVac_formulas_recovered_add
        else:
            inverse_mask += [False]
    xVac_formulas = xVac_formulas[inverse_mask]
    print(xVac_formulas)
    print(xVac_formulas_recovered)

    # Vacancy-based visual
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(5, 5), sharey='row', gridspec_kw={'height_ratios': [3, 1]})
    plt.suptitle(alloy_formulas[id_], x=0.15, y=0.95, fontweight="bold")
    for mode_id, formulas in enumerate([xVac_formulas_recovered, xVac_formulas]):
        tag = 'w/o vacancies' if mode_id == 0 else 'with S/Se vacancy'
        for form in formulas:
            axs[0, mode_id].plot(temperatures, 1000 * deltaG_table.loc[form][temperatures], color=Me_X_vac_2color[tag], alpha=0.2)
        axs[0, mode_id].scatter(np.zeros(len(formulas)), 1000 * res_table_plot.loc[formulas]['DFTminEhull'], color=Me_X_vac_2color[tag], marker='x')
        axs[0, mode_id].set_title(f'{tag}')
        axs[0, mode_id].grid(alpha=0.3)
        axs[0, mode_id].set_xlabel('T, K')
        axs[1, mode_id].hist(1000 * deltaG_table.loc[formulas]['∆Gdrop,eV/atom'], bins=1000 * np.linspace(-0.05, 0, 21), 
                             color=Me_X_vac_2color[tag], edgecolor='grey')
        axs[1, mode_id].grid(alpha=0.3)
        axs[1, mode_id].set_xlabel('${\Delta}G$ drop (0-2000K),\nmeV/atom')    
    axs[0, 0].set_ylabel('${\Delta}G$ with $T{\Delta}S_{conf}$\nabove 0 K hull, meV/atom')
    axs[1, 0].set_ylabel('Number of\ncompositions')
    plt.tight_layout()
    plt.show()